# Tạo và Nạp Bảng Chiều Đơn Vị Kinh Doanh DIM_BUSINESS_UNIT (Problem Statement 5)

Notebook này thực hiện tạo và nạp dữ liệu cho bảng Dimension chuẩn (`dim_business_unit`) trong Data Warehouse theo kiến trúc Star Schema:
- **Nguồn dữ liệu:** `Problem/ProblemStatement5/unit_kpi.csv` (98 đơn vị: 42 Văn phòng + 56 Đại lý).
- **Đặc tả Schema:** Gồm 11 cột chuẩn hóa (khóa chính `unit_key`, định danh `unit_id`, tên đơn vị, loại hình, địa bàn và các trường KPI/Rank/Đề xuất 2 trụ cột).
- **Mục tiêu:** Tách thực thể `dim_business_unit` độc lập, thiết lập khóa ngoại liên kết với `dim_geography` và `fact_orders` phục vụ phân tích đa chiều.

In [1]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

# 1. Đọc dữ liệu từ file CSV
def find_file(filename):
    for prefix in ["../../Problem/ProblemStatement5/", "../Problem/ProblemStatement5/", "Problem/ProblemStatement5/"]:
        path = os.path.join(prefix, filename)
        if os.path.exists(path):
            return path
    return filename

csv_path = find_file("unit_kpi.csv")
df_raw = pd.read_csv(csv_path)
print(f"Đã đọc {len(df_raw)} dòng từ {csv_path}")

# 2. Chuẩn hóa và tính toán KPI 2 trụ cột (ARPU + AOV) theo Tứ phân vị chuẩn xác
unit_dim = df_raw[['unit_id', 'unit_type', 'region', 'city', 'district']].copy()
unit_dim['unit_name'] = unit_dim['unit_type'] + ' ' + unit_dim['city'] + ' (' + unit_dim['district'] + ')'
unit_dim['unit_key'] = range(1, len(unit_dim) + 1)

# Tính điểm phần vị (Percentile) của 2 trụ cột giá trị
arpu_pctile = df_raw['revenue_per_customer'].rank(pct=True)
aov_pctile = df_raw['revenue_per_invoice'].rank(pct=True)
unit_dim['kpi_score'] = (((arpu_pctile + aov_pctile) / 2.0) * 100).round(2)

# Xếp hạng nội bộ theo loại đơn vị (Văn phòng: 1..42, Đại lý: 1..56)
unit_dim['overall_rank'] = unit_dim.groupby('unit_type')['kpi_score'].rank(ascending=False, method='min').astype(int)

# Thang đo Likert 4 mức theo Tứ phân vị chuẩn xác
q1 = unit_dim['kpi_score'].quantile(0.25)
q2 = unit_dim['kpi_score'].quantile(0.50)
q3 = unit_dim['kpi_score'].quantile(0.75)

def get_kpi_label(score):
    if score >= q3:
        return '4. Xuất sắc'
    elif score >= q2:
        return '3. Tốt'
    elif score >= q1:
        return '2. Trung bình'
    else:
        return '1. Thấp'

unit_dim['unit_performance_kpi'] = unit_dim['kpi_score'].apply(get_kpi_label)

# Đề xuất chiến lược tối ưu mạng lưới theo mô hình đơn vị
def get_rec(row):
    group = row['unit_performance_kpi']
    u_type = row['unit_type']
    if group == '4. Xuất sắc':
        return 'Ưu tiên mở rộng & Nhân rộng mô hình'
    elif group == '3. Tốt':
        return 'Duy trì & Tối ưu chi phí vận hành'
    elif group == '2. Trung bình':
        return 'Thúc đẩy bán chéo & Tăng AOV'
    else:
        if u_type == 'Văn phòng':
            return 'Rà soát chi phí & Chuyển đổi sang Đại lý'
        else:
            return 'Cắt giảm chi phí & Tái cơ cấu'

unit_dim['recommendation'] = unit_dim.apply(get_rec, axis=1)

# Sắp xếp đúng 11 cột chuẩn của CSDL
ordered_cols = [
    'unit_key', 'unit_id', 'unit_name', 'unit_type', 'region', 'city', 'district',
    'kpi_score', 'overall_rank', 'unit_performance_kpi', 'recommendation'
]
unit_dim = unit_dim[ordered_cols]
print("Dữ liệu mẫu DIM_BUSINESS_UNIT chuẩn 11 cột:")
print(unit_dim.head())
print(f"Kích thước: {unit_dim.shape}")


Đã đọc 98 dòng từ ../../Problem/ProblemStatement5/unit_kpi.csv
Dữ liệu mẫu DIM_BUSINESS_UNIT:


,unit_key,unit_id,unit_name,unit_type,region,city,district,total_customers,customer_percentile_in_city,customer_rank_in_city
0,1,Đại lý | East | Ha Long | District #03,Đại lý Ha Long (District #03),Đại lý,East,Ha Long,District #03,344,0.947368,2.0
1,2,Đại lý | East | Phu Ly | District #01,Đại lý Phu Ly (District #01),Đại lý,East,Phu Ly,District #01,355,0.947368,2.0
2,3,Đại lý | East | Uong Bi | District #09,Đại lý Uong Bi (District #09),Đại lý,East,Uong Bi,District #09,297,0.947368,2.0
3,4,Đại lý | West | Long Xuyen | District #39,Đại lý Long Xuyen (District #39),Đại lý,West,Long Xuyen,District #39,196,0.857143,2.0
4,5,Đại lý | East | Bac Ninh | District #03,Đại lý Bac Ninh (District #03),Đại lý,East,Bac Ninh,District #03,291,0.947368,2.0


Kích thước: (98, 10)


In [2]:
# 3. Kết nối Microsoft SQL Server từ db_engine
from db_engine import get_raw_connection

conn = get_raw_connection()
cur = conn.cursor()
cur.fast_executemany = True

# 4. Đảm bảo bảng DIM_BUSINESS_UNIT tồn tại đúng 11 cột chuẩn theo StarSchema.sql
cur.execute('''
IF OBJECT_ID(N'dbo.dim_business_unit', N'U') IS NULL
CREATE TABLE dbo.dim_business_unit (
    unit_key INT PRIMARY KEY,
    unit_id NVARCHAR(150) UNIQUE NOT NULL,
    unit_name NVARCHAR(150) NOT NULL,
    unit_type NVARCHAR(50) NOT NULL,
    region NVARCHAR(50) NOT NULL,
    city NVARCHAR(100) NOT NULL,
    district NVARCHAR(100) NOT NULL,
    kpi_score DECIMAL(5,2) NULL,
    overall_rank INT NULL,
    unit_performance_kpi NVARCHAR(50) NULL,
    recommendation NVARCHAR(100) NULL
);
''')

# 5. Nạp dữ liệu vào DIM_BUSINESS_UNIT
cur.execute('DELETE FROM dbo.dim_business_unit;')

insert_sql = '''
INSERT INTO dbo.dim_business_unit (
    unit_key, unit_id, unit_name, unit_type, region, city, district,
    kpi_score, overall_rank, unit_performance_kpi, recommendation
) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
'''
rows_to_insert = [
    (
        int(r['unit_key']),
        str(r['unit_id']),
        str(r['unit_name']),
        str(r['unit_type']),
        str(r['region']),
        str(r['city']),
        str(r['district']),
        float(r['kpi_score']) if pd.notnull(r['kpi_score']) else None,
        int(r['overall_rank']) if pd.notnull(r['overall_rank']) else None,
        str(r['unit_performance_kpi']) if pd.notnull(r['unit_performance_kpi']) else None,
        str(r['recommendation']) if pd.notnull(r['recommendation']) else None
    )
    for _, r in unit_dim.iterrows()
]

cur.executemany(insert_sql, rows_to_insert)
conn.commit()
print(f"Đã nạp thành công {len(unit_dim)} đơn vị vào DIM_BUSINESS_UNIT trên SQL Server chuẩn 11 cột!")
conn.close()


Đã nạp thành công 98 đơn vị vào DIM_BUSINESS_UNIT trên SQL Server!
